# Week 1–2: Bengali SLM Baseline Evaluation
### Research: English vs Bengali Instruction Tuning in Small Language Models

**Goal:** Establish zero-shot baseline accuracy on Bengali tasks — NO fine-tuning yet.  
**Models:** Gemma-3-1B-it, TinyLlama-1.1B  
**Tasks:** Sentiment Classification + Natural Language Inference (NLI)  
**These numbers are your anchor — everything in Week 3–8 is compared against them.**

---

## Step 0: Install Dependencies

In [ ]:
!pip install transformers datasets accelerate bitsandbytes -q

## Step 1: Imports and Configuration

In [10]:
from huggingface_hub import login
login("YOUR_HF_TOKEN_HERE")
print("✅")

✅


In [11]:
import torch
import json
import time
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---- CONFIGURATION ----
MODELS = {
    "gemma-3-1b": "google/gemma-3-1b-it",
    "tinyllama":  "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
}

SAMPLE_SIZE = 5  # 200 samples per task — fast enough on free T4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print("✅")

Device: cuda
GPU: Tesla P100-PCIE-16GB
✅


## Step 2: Model Loader and Inference Function

In [34]:
def load_model(model_name):
    print(f"Loading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16,
        device_map="auto"
    )
    model.eval()
    params = sum(p.numel() for p in model.parameters()) / 1e9
    print(f"Loaded. Parameters: {params:.2f}B")
    return tokenizer, model


def run_inference(prompt, tokenizer, model, max_new_tokens=20):
    messages = [{"role": "user", "content": prompt}]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(formatted, return_tensors="pt",
                       truncation=True, max_length=512).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip().lower()

print("✅")

✅


## Step 3: Task 1 — Sentiment Classification (SentNoB)
**Dataset:** Bengali sentiment, 3 labels: positive / negative / neutral

In [50]:
def build_sentiment_prompt(text):
    return f"""You are a sentiment classifier. Read this Bengali text carefully.

Bengali Text: {text}

Examples:
- "আমি খুব খুশি" → positive
- "এটা খুব খারাপ" → negative  
- "আবহাওয়া স্বাভাবিক" → neutral

What is the sentiment of the given text?
Reply with one word only"""
# ^^^ negative listed FIRST now — breaks position bias

def parse_sentiment(output):
    output = output.lower().strip()
    if "positive" in output: return "positive"
    if "negative" in output: return "negative"
    if "neutral"  in output: return "neutral"
    if "ইতিবাচক" in output: return "positive"
    if "নেতিবাচক" in output: return "negative"
    if "নিরপেক্ষ" in output: return "neutral"
    return "unknown"


def evaluate_sentiment(tokenizer, model):
    print("\n--- Loading Sentiment Dataset (SentNoB) ---")
    dataset = load_dataset("sadi2oo3q/SentNoB", split="test")

    label_map = {0: "negative", 1: "neutral", 2: "positive"}
    samples = dataset.shuffle(seed=42).select(range(min(SAMPLE_SIZE, len(dataset))))
    correct, unknowns, results = 0, 0, []

    print(f"Evaluating {len(samples)} samples...")
    for i, item in enumerate(samples):
        text       = item["Data"]
        true_label = label_map[item["Label"]]
        output     = run_inference(build_sentiment_prompt(text), tokenizer, model)
        predicted  = parse_sentiment(output)
        print(f"output: {predicted}")
        print(f"truth {true_label}")

        if predicted == "unknown": unknowns += 1
        if predicted == true_label: correct += 1

        results.append({
            "text": text[:80], "true": true_label,
            "predicted": predicted, "correct": predicted == true_label,
            "raw_output": output[:50]
        })

        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{len(samples)}] Accuracy so far: {correct/(i+1)*100:.1f}%")

    acc = correct / len(samples) * 100
    print(f"\nSentiment Accuracy : {acc:.2f}%")
    print(f"Unknown predictions: {unknowns}/{len(samples)}")
    return acc, results

print("✅")

✅


## Step 4: Task 2 — Natural Language Inference (xnli_bn)
**Dataset:** Bengali NLI, 3 labels: entailment / neutral / contradiction

In [36]:
# def build_hate_prompt(text):
#     return f"""You are detecting hate speech in Bengali text.

# Text: {text}

# Does this text contain hate speech? Reply with exactly one word: hate or not_hate.
# Answer:"""


# def parse_hate(output):
#     if "not_hate" in output or "not hate" in output:
#         return "not_hate"
#     if "hate" in output:
#         return "hate"
#     return "unknown"


# def evaluate_hate_speech(tokenizer, model):
#     print("\n--- Loading Hate Speech Dataset (Bengali) ---")
#     dataset = load_dataset("sadi2oo3q/Bengali_hate_speech_dataset", split="test")

#     # label: 0 = not_hate, 1 = hate
#     label_map = {0: "not_hate", 1: "hate"}

#     samples = dataset.shuffle(seed=42).select(range(min(SAMPLE_SIZE, len(dataset))))

#     correct, unknowns, results = 0, 0, []

#     print(f"Evaluating {len(samples)} samples...")
#     for i, item in enumerate(samples):
#         text       = item["sentence"]           # correct column name
#         true_label = label_map[item["hate"]]    # correct column name

#         output    = run_inference(build_hate_prompt(text), tokenizer, model)
#         predicted = parse_hate(output)

#         if predicted == "unknown": unknowns += 1
#         if predicted == true_label: correct += 1

#         results.append({
#             "text":       text[:80],
#             "category":   item["category"],
#             "true":       true_label,
#             "predicted":  predicted,
#             "correct":    predicted == true_label,
#             "raw_output": output[:50]
#         })

#         if (i + 1) % 50 == 0:
#             print(f"  [{i+1}/{len(samples)}] Accuracy so far: {correct/(i+1)*100:.1f}%")

#     acc = correct / len(samples) * 100
#     print(f"\nHate Speech Accuracy: {acc:.2f}%")
#     print(f"Unknown predictions : {unknowns}/{len(samples)}")
#     return acc, results

# print("✅")

In [52]:
# On Kaggle: Add Data → search "bengali-question-answering-dataset" by mayeesha
# Path: /kaggle/input/bengali-question-answering-dataset/

import json, os

def load_bengali_qa(path="/kaggle/input/datasets/mayeesha/bengali-question-answering-dataset"):
    files = [f for f in os.listdir(path) if f.endswith('.json')]
    print("Files found:", files)
    with open(os.path.join(path, files[0]), 'r', encoding='utf-8') as f:
        data = json.load(f)
    # SQuAD format: data > paragraphs > qas > question/answers
    samples = []
    for article in data['data']:
        for para in article['paragraphs']:
            context = para['context']
            for qa in para['qas']:
                if not qa.get('is_impossible', False) and qa['answers']:
                    samples.append({
                        'context':  context,
                        'question': qa['question'],
                        'answer':   qa['answers'][0]['text']
                    })
    return samples


def build_qa_prompt(context, question):
    return f"""Read the Bengali passage and answer the question.

Passage: {context[:400]}

Question: {question}

Give a short answer in Bengali.
Answer:"""


def evaluate_qa(tokenizer, model):
    print("\n--- Loading Bengali QA Dataset (SQuAD-BN) ---")
    samples_raw = load_bengali_qa()

    import random
    random.seed(42)
    samples = random.sample(samples_raw, min(SAMPLE_SIZE, len(samples_raw)))

    partial_match, exact_match, results = 0, 0, []

    print(f"Evaluating {len(samples)} samples...")
    for i, item in enumerate(samples):
        true_answer = item['answer'].lower().strip()
        output      = run_inference(
            build_qa_prompt(item['context'], item['question']),
            tokenizer, model, max_new_tokens=30
        )
        predicted = output.strip().lower()

        # Partial match — if any word of true answer appears in output
        is_partial = any(word in predicted for word in true_answer.split() if len(word) > 2)
        is_exact   = true_answer in predicted

        if is_partial: partial_match += 1
        if is_exact:   exact_match += 1

        results.append({
            "question":  item['question'][:60],
            "true":      true_answer[:60],
            "predicted": predicted[:60],
            "partial":   is_partial,
            "exact":     is_exact
        })


        print(f"output : {predicted}")
        print(f"correct : {true_answer}")
        

        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{len(samples)}] Partial: {partial_match/(i+1)*100:.1f}% | Exact: {exact_match/(i+1)*100:.1f}%")

    partial_acc = partial_match / len(samples) * 100
    exact_acc   = exact_match  / len(samples) * 100
    print(f"\nPartial Match : {partial_acc:.2f}%")
    print(f"Exact Match   : {exact_acc:.2f}%")
    return partial_acc, results

## Step 5: Run All Evaluations
⏱️ Expected time: ~30–45 min per model on free T4

In [38]:
all_results = {}

for model_key, model_path in MODELS.items():
    print(f"\n{'='*60}")
    print(f"EVALUATING: {model_key}")
    print(f"{'='*60}")

    start = time.time()
    tokenizer, model = load_model(model_path)

print("✅")


EVALUATING: gemma-3-1b
Loading google/gemma-3-1b-it...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loaded. Parameters: 1.00B

EVALUATING: tinyllama
Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded. Parameters: 1.10B
✅


In [53]:
# sent_acc, sent_results = evaluate_sentiment(tokenizer, model)
# nli_acc,  nli_results  = evaluate_nli(tokenizer, model)
qa_partial, qa_results = evaluate_qa(tokenizer, model)


# elapsed = time.time() - start

# all_results[model_key] = {
#     "sentiment_accuracy": round(sent_acc, 2),
#     "nli_accuracy":       round(nli_acc, 2),
#     "time_minutes":       round(elapsed / 60, 1),
#     "sentiment_samples":  sent_results[:20],
#     "nli_samples":        nli_results[:20],
# }

# # Free GPU memory before next model
# del model, tokenizer
# torch.cuda.empty_cache()
# print(f"\nDone in {elapsed/60:.1f} minutes")


--- Loading Bengali QA Dataset (SQuAD-BN) ---
Files found: ['train_bangla_samples_fixed_preprocessed.json', 'preprocessed_translations_final_fixed.json', 'valid_bangla_samples_fixed_preprocessed.json']
Evaluating 5 samples...
output : একটি অভিযোগ আইনের
correct : তে ব্যাপক আ
output : ি শিশুশ্রমের কারণে এই ��
correct : বাংলাদেশ, ব্রাজিল, চীন, মিশর,
output : উটেটিং এর অভিযোগ
correct : পরিবেশগত অ
output : িশ গ্রামীণ লোকের চিঠি ��
correct : তার পরিবার
output : য়ানরা উপস্থিতি প্রতিবেদনে
correct : উত্তর রাশিয়ানরা

Partial Match : 0.00%
Exact Match   : 0.00%


## Step 6: Results Summary Table
**These are your anchor numbers — save them.**

In [ ]:
print("\n" + "="*60)
print("   BASELINE RESULTS — ZERO-SHOT (No Fine-Tuning)")
print("="*60)
print(f"{'Model':<20} {'Sentiment':>12} {'NLI':>10} {'Time':>10}")
print("-"*56)
for model_key, res in all_results.items():
    print(f"{model_key:<20} {res['sentiment_accuracy']:>11.2f}% "
          f"{res['nli_accuracy']:>9.2f}% "
          f"{res['time_minutes']:>8.1f}m")

print("\n*** These numbers are your Week 3-8 comparison baseline ***")

# Save results
with open("baseline_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("Saved to baseline_results.json")

## Notes & Realistic Expectations

| Model | Expected Sentiment | Expected NLI | Notes |
|---|---|---|---|
| Gemma-3-1B-it | 45–60% | 35–50% | Better alignment, handles Bengali better |
| TinyLlama-1.1B | 30–45% | 30–38% | Near random on NLI (3-class = 33% random) |

**If you see many `unknown` predictions:**  
The model is answering in Bengali instead of English label words. Report this to fix the parser.

**Random baseline:**
- Sentiment (3-class): 33.3%
- NLI (3-class): 33.3%

Anything above this means the model has some Bengali understanding.